# Memory and Retrieval

The following is a toy example of an agent that reads and writes to an external data source. One characteristic of retrieval systems is that the entire process is **stateless**, e.g. it cannot learn from interactions. Retrieval systems generally involve queries to an external data source, then adding the response to the current model context.

For the following example, the LLM also writes to the same memory store hence affecting future generation states. It follows that this system is **stateful**. We can think of the external memory store as the **long-term memory** of the system. On the other hand, LLMs naturally have **short-term memory** in the form of its context. The architecture is shown in @fig-retrieval-system.

![The LLM expresses the intent to write to the memory store via the structured output. Then, it is up to the main program to perform the actual writing. This allows hooks like [guardrails](https://cookbook.openai.com/examples/how_to_use_guardrails) to be applied before executing the function. Note that the retrieval happens prior to LLM processing. It would be nice to have the LLM read the entire filestore but this becomes more expensive as the memory store grows. In practice, information retrieval techniques such as TF-IDF and embedding similarity can be used.
](./img/retrieval-system.png){#fig-retrieval-system}

**Memory store.** Memories are saved as dictionaries `{"tag": <tag>, "fact": <fact>}`. The following objects work around this definition. A retrieval function is defined as a method of the memory store class which gets relevant memory items based on *keyword search*. Hence, we have the following function for extracting keywords:

In [ ]:
import string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

stop_words = set(stopwords.words("english"))

def tokenize(text):
    """Tokenize into words then remove stopwords."""
    tokens = word_tokenize(text.lower())
    filtered_tokens = [word for word in tokens if word not in stop_words and word not in string.punctuation]
    return filtered_tokens

text = "Punk is a pre-trained unsupervised machine learning model for tokenization. It's one of the most crucial and widely used components in the NLTK library."
print("Original:", text)
print("Filtered:", tokenize(text))

Original: Punk is a pre-trained unsupervised machine learning model for tokenization. It's one of the most crucial and widely used components in the NLTK library.
Filtered: ['punk', 'pre-trained', 'unsupervised', 'machine', 'learning', 'model', 'tokenization', "'s", 'one', 'crucial', 'widely', 'used', 'components', 'nltk', 'library']


In [ ]:
import json
from typing import List
from openai import OpenAI
from pydantic import BaseModel


class MemoryItem(BaseModel):
    tag: str
    fact: str
    reason: str

class MemoryResponse(BaseModel):
    items: List[MemoryItem]


class MemoryStore:
    def __init__(self, path="memory.json"):
        """Load memory from JSON file in local path."""
        self.path = path
        self.data = []
        self.tags = set()
        self.load()

    def load(self):
        try:
            self.data = json.load(open(self.path))
        except FileNotFoundError:
            self.reset()

    def save(self):
        with open(self.path, "w") as f:
            json.dump(self.data, f, indent=2)

    def reset(self):
        self.data = []
        self.tags = set()
        self.save()
    
    def add(self, item: MemoryItem):
        tag, fact = item.tag, item.fact
        self.tags.add(tag)
        self.data.append({"tag": tag, "fact": fact})

    def __len__(self):
        return len(self.data)
    
    def retrieve(self, query: str, topk: int = 3) -> List[dict]:
        """Simple keyword-based retrieval."""
        
        query_words = tokenize(query)
        retrieved = []
        
        for memory in reversed(self.data):  # <1>
            tag, fact = memory["tag"], memory["fact"]
            fact = ' '.join(tokenize(fact))
            memory_text = f"{tag} {fact}".lower()

            for word in query_words:
                if word in memory_text: # <2>
                    retrieved.append(memory)
                    break
            
            if len(retrieved) == topk:
                break
        
        return retrieved


client = OpenAI()
mem = MemoryStore()

1. More recent = more relevant.
2. Substring check. e.g. `'commute' in 'commute_experience'` evaluates to `True`.

Next, we define the **generation step** and the **write step**:

In [ ]:
prompt_template = lambda memories, tags: f"""
You are an assistant that processes daily user logs. For each log, extract a concise, 
factual summary of what happened. Each summary should be atomic, standalone, and 
likely useful for future interactions. Assign a relevant `tag` to each summary (`fact`) 
before saving it to memory. 

The following are relevant entries (based on the current input) in the Memory Store:
{memories}

The following are the current tags:
{tags}

**GUIDELINES:**

1. **EXTRACT ATOMIC FACTS:**
    - Break down information into the smallest meaningful, self-contained units.
    - **Good**: "User's favorite programmer is Jon Blow."
    - **Bad**: "User mentioned their favorite programmer is Jon Blow who is a famous game programmer" (This has two facts.)
    - The `fact` must be a concise, direct paraphrase of the fact. Remove conversational fluff.
    - **Good Info:** "User's favorite city is Tokyo"
    - **Bad Info:** "The user stated that if they had to pick a favorite city, they think it would be Tokyo."

2.  **TAG EFFECTIVELY:**
    - **Format:** Prefer generic, descriptive tags in `snake_case`.
    - **Simple:** Prefer simple tags. Choose `commute` is better than `commute_experience`.
    - **Reuse:** Strongly prefer existing tags. Create a new tag only if necessary.
    - An example: For "I really enjoy hiking in the Alps every summer," a good tag is `hobby` or `outdoor_activity`.
    
4.  **EVALUATE & DECIDE:**
    - It is acceptable to save zero logs from an input if nothing is meaningfully new or relevant.
    - Save multiple logs if the user provides multiple distinct pieces of information.
    - Each memory item should make sense on its own. There should be no dependence between separate logs.
"""


def capture_memorable_facts(user_log: str, topk: int=3) -> MemoryResponse:
    """Generate memorable facts from log and write them to memory."""

    # generate memorable facts based on relevant items
    retrieved_memories = mem.retrieve(user_log, topk)
    current_tags = list(mem.tags)
    system_prompt = prompt_template(retrieved_memories, current_tags)

    response = client.chat.completions.parse(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_log},
        ],
        response_format=MemoryResponse,
    )
    
    # save items to memory store
    out = response.choices[0].message.parsed
    for item in out.items:
        mem.add(item)

    mem.save()
    return out

Examples:

In [ ]:
import pandas as pd

logs = [
    "Woke up later than usual because I forgot to set an alarm, rushed through a quick shower, skipped coffee, and still managed to leave for work on time.",
    "Traffic was unusually light, but halfway through I realized I left my ID at home, debated turning back, and decided to just explain at the office front desk.",
    "Took the train, found no seats since it was packed, but struck up a short conversation with a stranger about the book they were reading while we both stood.",
    "Stopped by the bakery, picked up bread for the team, and noticed they had a new seasonal pastry that tempted me but I decided to pass.",
    "Opened my email first thing at the office, skimmed through a pile of routine messages, flagged one urgent client request, and forwarded it to the team lead.",
    "Woah! While crossing the street a man suddenly darted into traffic, cars honked, everyone gasped, and I stood frozen for a moment before hurrying away still shaken.",
    "Listened to a podcast while walking to the subway, half distracted by construction noise on the street, and made a mental note to check out the book they recommended.",
    "Grabbed a pen from my drawer because mine ran out of ink, ended up reorganizing the entire drawer, and discovered an old sticky note with a reminder I had long forgotten."
]

items = []
for log in logs:
    memory_items = capture_memorable_facts(log).items
    for item in memory_items:
        d = item.model_dump()
        d["text"] = log
        items.append(d)

df_resp = pd.DataFrame(items)
mem.reset()

The agent decides whether to reuse a tag or create a new one based on the data:

In [ ]:
#| code-fold: true
import warnings
warnings.simplefilter("ignore")
pd.set_option('display.max_colwidth', None)

print(f"({len(logs)} total logs, {len(df_resp)} facts saved, {len(df_resp.tag.unique())} tags)")
print("tags:")
pprint(list(df_resp.tag.unique()))
df_resp[["tag", "fact", "text", "reason"]]

(8 total logs, 10 facts saved, 8 tags)
tags:
['commute',
 'food_purchase',
 'work_task',
 'unexpected_incident',
 'media_consumption',
 'book_interest',
 'workspace_organization',
 'personal_discovery']


,tag,fact,text,reason
0,commute,"User managed to leave for work on time despite waking up late, skipping coffee, and rushing through a shower.","Woke up later than usual because I forgot to set an alarm, rushed through a quick shower, skipped coffee, and still managed to leave for work on time.",This aligns with existing commute experiences and contributes to understanding daily schedules.
1,commute,User forgot their ID at home but decided to explain at the office front desk instead of turning back.,"Traffic was unusually light, but halfway through I realized I left my ID at home, debated turning back, and decided to just explain at the office front desk.","This fact complements the existing information about the commute, noting the decision-making process and potential impact."
2,food_purchase,User picked up bread for the team at the bakery.,"Stopped by the bakery, picked up bread for the team, and noticed they had a new seasonal pastry that tempted me but I decided to pass.","This log details a specific purchase made by the user, which may be relevant for future preferences or routines."
3,work_task,User opened their email first thing at the office and forwarded an urgent client request to the team lead.,"Opened my email first thing at the office, skimmed through a pile of routine messages, flagged one urgent client request, and forwarded it to the team lead.",The log provides a summary of a specific task conducted by the user as part of their work routine.
4,unexpected_incident,"User witnessed a man dart into traffic, causing cars to honk and bystanders to gasp, which left them shaken.","Woah! While crossing the street a man suddenly darted into traffic, cars honked, everyone gasped, and I stood frozen for a moment before hurrying away still shaken.",This is a distinct and memorable experience regarding an unexpected and alarming incident.
5,media_consumption,User listened to a podcast while walking to the subway.,"Listened to a podcast while walking to the subway, half distracted by construction noise on the street, and made a mental note to check out the book they recommended.","This indicates what the user was doing during their walk, related to media consumption."
6,commute,User was half distracted by construction noise on the street while walking to the subway.,"Listened to a podcast while walking to the subway, half distracted by construction noise on the street, and made a mental note to check out the book they recommended.",This describes part of the user's experience during their commute.
7,book_interest,User made a mental note to check out a book recommended in a podcast.,"Listened to a podcast while walking to the subway, half distracted by construction noise on the street, and made a mental note to check out the book they recommended.",This indicates a new interest or intention to explore a book.
8,workspace_organization,User reorganized their drawer after retrieving a pen.,"Grabbed a pen from my drawer because mine ran out of ink, ended up reorganizing the entire drawer, and discovered an old sticky note with a reminder I had long forgotten.",Recording about workspace organization may provide context for future tasks or preferences.
9,personal_discovery,User discovered an old sticky note with a forgotten reminder in their drawer.,"Grabbed a pen from my drawer because mine ran out of ink, ended up reorganizing the entire drawer, and discovered an old sticky note with a reminder I had long forgotten.",Finding a forgotten reminder may impact user's future decisions or tasks.
